# Recommendation Systems

Recommendation systems predict user preferences to suggest relevant items. They power Netflix, Spotify, Amazon, YouTube, and most modern digital platforms.

## Types of Recommendation Systems

| Type | Approach | Cold Start? |
|------|----------|-------------|
| Collaborative Filtering | User-item interactions | Problem |
| Content-Based | Item features | No for items |
| Hybrid | Both | Mitigated |
| Knowledge-Based | Explicit rules/constraints | No |

---

## 1. Collaborative Filtering

Based on the idea: *"users who agreed in the past will agree in the future."*

### User-Based CF
Find similar users, predict rating based on their ratings:
$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in N(u)} \text{sim}(u,v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in N(u)} |\text{sim}(u,v)|}$$

**Cosine Similarity**:
$$\text{sim}(u, v) = \cos(\theta) = \frac{u \cdot v}{||u|| \cdot ||v||}$$

**Pearson Correlation**:
$$\text{sim}(u, v) = \frac{\sum_i (r_{ui}-\bar{r}_u)(r_{vi}-\bar{r}_v)}{\sqrt{\sum_i(r_{ui}-\bar{r}_u)^2}\sqrt{\sum_i(r_{vi}-\bar{r}_v)^2}}$$

### Item-Based CF
Find similar items, predict based on user's other ratings:
$$\hat{r}_{ui} = \frac{\sum_{j \in N(i)} \text{sim}(i,j) \cdot r_{uj}}{\sum_{j \in N(i)} |\text{sim}(i,j)|}$$

---

## 2. Matrix Factorization

Decompose the user-item rating matrix $R$ into two low-rank matrices:
$$R \approx P \times Q^T, \quad \hat{r}_{ui} = p_u^T q_i$$

where $P \in \mathbb{R}^{m \times k}$ (user factors) and $Q \in \mathbb{R}^{n \times k}$ (item factors).

**Objective with regularization**:
$$\min_{P,Q} \sum_{(u,i)\in\mathcal{K}} (r_{ui} - p_u^T q_i)^2 + \lambda(||p_u||^2 + ||q_i||^2)$$

**SGD Updates**:
$$e_{ui} = r_{ui} - p_u^T q_i$$
$$p_u \leftarrow p_u + \eta(e_{ui} q_i - \lambda p_u)$$
$$q_i \leftarrow q_i + \eta(e_{ui} p_u - \lambda q_i)$$

**With biases**:
$$\hat{r}_{ui} = \mu + b_u + b_i + p_u^T q_i$$

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')

# Create synthetic ratings matrix
np.random.seed(42)
n_users, n_items = 50, 30
# Sparse ratings (0 = not rated)
ratings = np.zeros((n_users, n_items))
for u in range(n_users):
    n_rated = np.random.randint(5, 15)
    items = np.random.choice(n_items, n_rated, replace=False)
    ratings[u, items] = np.random.randint(1, 6, n_rated).astype(float)

print(f"Rating matrix shape: {ratings.shape}")
print(f"Sparsity: {(ratings==0).sum() / ratings.size:.2%}")

# User-based collaborative filtering
class UserBasedCF:
    def __init__(self, k=10):
        self.k = k
    
    def fit(self, R):
        self.R = R.copy()
        # Replace 0 with NaN for mean calculation
        R_nan = R.copy().astype(float)
        R_nan[R_nan == 0] = np.nan
        self.user_means = np.nanmean(R_nan, axis=1)
        # Compute cosine similarity
        self.sim = cosine_similarity(R)
        return self
    
    def predict(self, user, item):
        # Users who rated this item
        rated = np.where(self.R[:, item] > 0)[0]
        if len(rated) == 0:
            return self.user_means[user]
        # Top-k similar users who rated item
        sims = self.sim[user, rated]
        top_k = rated[np.argsort(sims)[::-1][:self.k]]
        top_sims = self.sim[user, top_k]
        if top_sims.sum() == 0:
            return self.user_means[user]
        # Weighted average
        numerator = sum(top_sims[i] * (self.R[v, item] - self.user_means[v])
                       for i, v in enumerate(top_k))
        return self.user_means[user] + numerator / (np.abs(top_sims).sum() + 1e-9)

cf = UserBasedCF(k=10).fit(ratings)
print(f"\nUser 0, Item 5 predicted rating: {cf.predict(0, 5):.2f}")

# Matrix Factorization from scratch
class MatrixFactorization:
    def __init__(self, n_factors=10, lr=0.01, reg=0.01, n_epochs=100):
        self.k = n_factors; self.lr = lr; self.reg = reg; self.n_epochs = n_epochs
    
    def fit(self, R):
        m, n = R.shape
        self.P = np.random.normal(0, 0.1, (m, self.k))
        self.Q = np.random.normal(0, 0.1, (n, self.k))
        self.bu = np.zeros(m); self.bi = np.zeros(n)
        self.mu = R[R > 0].mean()
        users, items = np.where(R > 0)
        
        for epoch in range(self.n_epochs):
            total_loss = 0
            for u, i in zip(users, items):
                e = R[u,i] - (self.mu + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i])
                self.bu[u] += self.lr * (e - self.reg * self.bu[u])
                self.bi[i] += self.lr * (e - self.reg * self.bi[i])
                self.P[u] += self.lr * (e * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (e * self.P[u] - self.reg * self.Q[i])
                total_loss += e**2
            if (epoch+1) % 20 == 0:
                rmse = np.sqrt(total_loss / len(users))
                print(f"Epoch {epoch+1}: RMSE = {rmse:.4f}")
        return self
    
    def predict(self, u, i):
        return np.clip(self.mu + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i], 1, 5)
    
    def recommend(self, u, n=5, already_rated=None):
        scores = [self.predict(u, i) for i in range(self.Q.shape[0])]
        if already_rated is not None:
            for i in already_rated: scores[i] = -np.inf
        return np.argsort(scores)[::-1][:n]

mf = MatrixFactorization(n_factors=10, lr=0.005, reg=0.02, n_epochs=60).fit(ratings)
rated_by_user0 = np.where(ratings[0] > 0)[0]
recs = mf.recommend(0, n=5, already_rated=rated_by_user0)
print(f"\nTop-5 recommendations for user 0: {recs}")

Rating matrix shape: (50, 30)
Sparsity: 66.47%

User 0, Item 5 predicted rating: 3.61
Epoch 20: RMSE = 1.3196


Epoch 40: RMSE = 1.2212
Epoch 60: RMSE = 1.0122

Top-5 recommendations for user 0: [21  3  7 19 26]


---

## 3. Neural Collaborative Filtering (NCF)

Replaces dot product with a neural network for more expressive interactions:

$$\hat{y}_{ui} = f(p_u, q_i | \Theta)$$

**GMF** (Generalized Matrix Factorization):
$$\hat{y}_{ui} = \sigma(h^T(p_u \odot q_i))$$

**MLP component**:
$$z_1 = \phi_1(p_u^{MLP}, q_i^{MLP}) = [p_u^{MLP} \; ; \; q_i^{MLP}]$$
$$\phi_2(z_1) = a_2(W_2^T z_1 + b_2), \quad ... \quad \hat{y}_{ui} = \sigma(h^T \phi_L(z_{L-1}))$$

**NeuMF** (Neural Matrix Factorization) concatenates GMF and MLP outputs:
$$\hat{y}_{ui} = \sigma\left(h^T \begin{bmatrix} p_u^{GMF} \odot q_i^{GMF} \\ \phi_L^{MLP}(z_{L-1}) \end{bmatrix}\right)$$

## 4. Evaluation Metrics

**Precision@K**: Fraction of top-K recommendations that are relevant:
$$\text{Precision@K} = \frac{|\text{Relevant} \cap \text{Top-K}|}{K}$$

**Recall@K**:
$$\text{Recall@K} = \frac{|\text{Relevant} \cap \text{Top-K}|}{|\text{Relevant}|}$$

**NDCG@K** (Normalized Discounted Cumulative Gain):
$$\text{DCG@K} = \sum_{i=1}^K \frac{\text{rel}_i}{\log_2(i+1)}, \quad \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

**MRR** (Mean Reciprocal Rank):
$$\text{MRR} = \frac{1}{|U|}\sum_{u=1}^{|U|} \frac{1}{\text{rank}_u}$$

**Hit Rate@K**: Fraction of users who have at least one relevant item in top-K.

In [2]:
# Evaluation metrics
def precision_at_k(recommended, relevant, k):
    top_k = set(recommended[:k])
    return len(top_k & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    top_k = set(recommended[:k])
    return len(top_k & set(relevant)) / max(len(relevant), 1)

def ndcg_at_k(recommended, relevant, k):
    top_k = recommended[:k]
    dcg = sum((1 / np.log2(i+2)) for i, item in enumerate(top_k) if item in relevant)
    idcg = sum((1 / np.log2(i+2)) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0

def mean_reciprocal_rank(recommended_lists, relevant_lists):
    rrs = []
    for recs, rels in zip(recommended_lists, relevant_lists):
        for rank, item in enumerate(recs, 1):
            if item in rels:
                rrs.append(1/rank); break
        else:
            rrs.append(0)
    return np.mean(rrs)

# Example evaluation
recommended = [3, 7, 1, 15, 9, 2, 20, 8, 12, 5]
relevant = {1, 3, 8, 15, 22}
K = 5
print(f"Precision@{K}: {precision_at_k(recommended, relevant, K):.4f}")
print(f"Recall@{K}:    {recall_at_k(recommended, relevant, K):.4f}")
print(f"NDCG@{K}:      {ndcg_at_k(recommended, relevant, K):.4f}")

# Surprise library (SVD-based)
try:
    from surprise import SVD, Dataset, Reader, accuracy
    from surprise.model_selection import cross_validate
    
    # Create surprise dataset from our matrix
    rows, cols = np.where(ratings > 0)
    data_df = pd.DataFrame({'user': rows, 'item': cols, 'rating': ratings[rows, cols]})
    reader = Reader(rating_scale=(1, 5))
    dataset = Dataset.load_from_df(data_df, reader)
    
    svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)
    results = cross_validate(svd, dataset, measures=['RMSE', 'MAE'], cv=3, verbose=True)
    print(f"\nSurprise SVD - RMSE: {results['test_rmse'].mean():.4f}")
except ImportError:
    print("Install Surprise: pip install scikit-surprise")

# Cold start problem strategies
print("\nCold Start Strategies:")
print("  New User: Ask initial preferences, use demographic data, popularity-based")
print("  New Item: Use content-based features, knowledge-based rules")
print("  Hybrid:   Combine CF + content-based, meta-learning")

Precision@5: 0.6000
Recall@5:    0.6000
NDCG@5:      0.6548
Evaluating RMSE, MAE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    1.4558  1.5280  1.4896  1.4911  0.0295  
MAE (testset)     1.2352  1.3237  1.2862  1.2817  0.0363  
Fit time          0.00    0.00    0.00    0.00    0.00    
Test time         0.00    0.00    0.00    0.00    0.00    

Surprise SVD - RMSE: 1.4911

Cold Start Strategies:
  New User: Ask initial preferences, use demographic data, popularity-based
  New Item: Use content-based features, knowledge-based rules
  Hybrid:   Combine CF + content-based, meta-learning


---

## 5. Advanced Methods Overview

### Wide & Deep Learning (Google)
- **Wide**: Memorization via cross-product feature interactions (logistic regression)
- **Deep**: Generalization via dense embeddings through deep network
- **Combined**: $P(Y=1|x) = \sigma(w_{wide}^T[x, \phi(x)] + w_{deep}^T a^{(l_f)} + b)$

### DeepFM
Replaces wide part with Factorization Machine:
$$\hat{y} = \sigma(y_{FM} + y_{DNN})$$

### Two-Tower Model
Separate encoders for users and items:
$$\hat{r}_{ui} = \cos(f_\theta(u), g_\phi(i))$$

Efficient for large-scale retrieval with approximate nearest neighbors (FAISS).

### LightGCN
Graph Convolutional Network for collaborative filtering:
$$e_u^{(k+1)} = \sum_{i \in N_u} \frac{1}{\sqrt{|N_u|}\sqrt{|N_i|}} e_i^{(k)}$$

Final: $e_u = \frac{1}{K+1}\sum_{k=0}^K e_u^{(k)}$

---

## Additional Learning Resources

### Papers
- **NCF** (Neural Collaborative Filtering): [https://arxiv.org/abs/1708.05031](https://arxiv.org/abs/1708.05031)
- **LightGCN**: [https://arxiv.org/abs/2002.02126](https://arxiv.org/abs/2002.02126)
- **Wide & Deep**: [https://arxiv.org/abs/1606.07792](https://arxiv.org/abs/1606.07792)
- **DeepFM**: [https://arxiv.org/abs/1703.04247](https://arxiv.org/abs/1703.04247)
- **Matrix Factorization (Koren)**: [https://ieeexplore.ieee.org/document/5197422](https://ieeexplore.ieee.org/document/5197422)

### Libraries
- **Surprise**: [https://surpriselib.com/](https://surpriselib.com/)
- **Implicit**: [https://github.com/benfred/implicit](https://github.com/benfred/implicit)
- **RecBole**: [https://recbole.io/](https://recbole.io/)
- **Merlin (NVIDIA)**: [https://github.com/NVIDIA-Merlin/Merlin](https://github.com/NVIDIA-Merlin/Merlin)

### Courses
- **Google Recommendation Systems**: [https://developers.google.com/machine-learning/recommendation](https://developers.google.com/machine-learning/recommendation)